In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_path = "/home/thanhdo/hub/deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
device = "cuda:1"
tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map={"": device},
)

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [8]:
messages = [
    {"role": "user", "content": "Hello bro"},
    {"role": "assistant", "content": "Wassup bro"},
    {"role": "user", "content": "What can I do for you bro?"},
    {"role": "assistant", "content": "Anything bro"},
]
bro = tokenizer.apply_chat_template(
    messages,
    tokenize              = False,
    add_generation_prompt = False,
    # return_tensors        = "pt",
)
print(bro)

<｜begin▁of▁sentence｜><｜User｜>Hello bro<｜Assistant｜>Wassup bro<｜end▁of▁sentence｜><｜User｜>What can I do for you bro?<｜Assistant｜>Anything bro<｜end▁of▁sentence｜>


In [9]:
tokenizer.chat_template

"{% if not add_generation_prompt is defined %}{% set add_generation_prompt = false %}{% endif %}{% set ns = namespace(is_first=false, is_tool=false, is_output_first=true, system_prompt='') %}{%- for message in messages %}{%- if message['role'] == 'system' %}{% set ns.system_prompt = message['content'] %}{%- endif %}{%- endfor %}{{bos_token}}{{ns.system_prompt}}{%- for message in messages %}{%- if message['role'] == 'user' %}{%- set ns.is_tool = false -%}{{'<｜User｜>' + message['content']}}{%- endif %}{%- if message['role'] == 'assistant' and message['content'] is none %}{%- set ns.is_tool = false -%}{%- for tool in message['tool_calls']%}{%- if not ns.is_first %}{{'<｜Assistant｜><｜tool▁calls▁begin｜><｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['function']['name'] + '\\n' + '```json' + '\\n' + tool['function']['arguments'] + '\\n' + '```' + '<｜tool▁call▁end｜>'}}{%- set ns.is_first = true -%}{%- else %}{{'\\n' + '<｜tool▁call▁begin｜>' + tool['type'] + '<｜tool▁sep｜>' + tool['fu

In [1]:
# Smoke test: hook-based extraction vs direct out.attentions.
# Keep MAX_TOKENS small so the unhooked forward (which materialises the
# full (L, 1, H, N, N) tuple) fits in VRAM.

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from attribscope.data.trajectory import load_dataset
from attribscope.data.context    import preprocess_context
from attribscope.reps.extract_attention import (
    extract_trajectory_attention, MODELS,
)

MODEL = "qwen3-4b"
SUBSET = ["algorithm-generated", "hand-crafted"][0]
MAX_TOKENS = 2048
DEVICE = "cuda:0"

# ── Load model with eager attention (required so attn probs are materialised)
tokenizer = AutoTokenizer.from_pretrained(MODELS[MODEL])
model = AutoModelForCausalLM.from_pretrained(
    MODELS[MODEL], 
    torch_dtype=torch.bfloat16,
    # device_map={"": DEVICE}, 
    device_map="auto",
    attn_implementation="eager",
)
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((2560,), eps=1e-06)
        (post_attention_layer

In [2]:
# ── Hooked path: run the implementation we want to validate ─────────────
traj = load_dataset("../data/ww", subset=SUBSET)[0]
flat = extract_trajectory_attention(
    traj, model, tokenizer,
    max_tokens=MAX_TOKENS, context="all", query_pool="mean",
)

In [4]:
flat.keys()

dict_keys(['1.raw_attn', '1.raw_attn_per_head', '1.attn_residual_mass', '1.ctx_indices', '2.raw_attn', '2.raw_attn_per_head', '2.attn_residual_mass', '2.ctx_indices', '3.raw_attn', '3.raw_attn_per_head', '3.attn_residual_mass', '3.ctx_indices', '4.raw_attn', '4.raw_attn_per_head', '4.attn_residual_mass', '4.ctx_indices', '5.raw_attn', '5.raw_attn_per_head', '5.attn_residual_mass', '5.ctx_indices'])

In [3]:
from attribscope.data.context import preprocess_context, iter_scoreable_steps
for s in iter_scoreable_steps(traj):
    enc = preprocess_context(traj, s, tokenizer, max_tokens=MAX_TOKENS, strategy="all")
    print(s, "ctx steps:", sorted(m for m in enc["step_tokens"] if m != s))

0 ctx steps: []
1 ctx steps: [0]
2 ctx steps: [0, 1]
3 ctx steps: [0, 1, 2]
4 ctx steps: [0, 1, 2, 3]
5 ctx steps: [0, 1, 2, 3, 4]


In [5]:
flat['5.raw_attn']

tensor([[0.0731, 0.0134, 0.1199, 0.0150, 0.3230],
        [0.1966, 0.0151, 0.1455, 0.0214, 0.2791],
        [0.3297, 0.0148, 0.0776, 0.0181, 0.2065],
        [0.3779, 0.0282, 0.0605, 0.0238, 0.1735],
        [0.4115, 0.0251, 0.0489, 0.0116, 0.1905],
        [0.3680, 0.0135, 0.0541, 0.0113, 0.1934],
        [0.3051, 0.0183, 0.0697, 0.0174, 0.1958],
        [0.0572, 0.0169, 0.0400, 0.0174, 0.0761],
        [0.0628, 0.0283, 0.0553, 0.0230, 0.1032],
        [0.0933, 0.0322, 0.0769, 0.0304, 0.1360],
        [0.0383, 0.0144, 0.0401, 0.0154, 0.1152],
        [0.0672, 0.0165, 0.0462, 0.0190, 0.1108],
        [0.0804, 0.0285, 0.0689, 0.0389, 0.1551],
        [0.0924, 0.0365, 0.0660, 0.0408, 0.1614],
        [0.0678, 0.0205, 0.0549, 0.0221, 0.1349],
        [0.0773, 0.0222, 0.0669, 0.0333, 0.1937],
        [0.0937, 0.0239, 0.0677, 0.0316, 0.1958],
        [0.0851, 0.0297, 0.0947, 0.0344, 0.2234],
        [0.0875, 0.0223, 0.0644, 0.0303, 0.2890],
        [0.0636, 0.0150, 0.0565, 0.0319, 0.2528],


In [6]:
STEP_IDX = 5   # or e.g. 17

available = sorted({int(k.split(".")[0]) for k in flat if k.endswith(".raw_attn")})
print(f"Steps with captured attention: {available}")

step_idx = available[0] if STEP_IDX is None else STEP_IDX
assert step_idx in available, f"step {step_idx} not in {available}"

hooked       = flat[f"{step_idx}.raw_attn_per_head"]
ctx_step_ids = flat[f"{step_idx}.ctx_indices"].tolist()

Steps with captured attention: [1, 2, 3, 4, 5]


In [7]:
# ── Reference path: same input, no hooks, manual reduction ──────────────
encoded     = preprocess_context(traj, step_idx, tokenizer,
                                 max_tokens=MAX_TOKENS, strategy="all")
input_ids   = encoded["input_ids"].to(DEVICE)
step_tokens = encoded["step_tokens"]

with torch.no_grad():
    out = model(input_ids, output_attentions=True, use_cache=False)

In [10]:
step_tokens.keys()

dict_keys([0, 1, 2, 3, 4, 5])

In [11]:
T_t = torch.tensor(step_tokens[step_idx], device=DEVICE)
ref = torch.zeros_like(hooked)
for l, A_l in enumerate(out.attentions):                          # A_l: (1, H, N, N)
    A_t = A_l[0].float().index_select(1, T_t).mean(dim=1)         # (H, N)
    for j, i in enumerate(ctx_step_ids):
        T_i = torch.tensor(step_tokens[i], device=DEVICE)
        ref[l, :, j] = A_t.index_select(1, T_i).sum(dim=1).cpu()

In [12]:
diff = (hooked - ref).abs()
print(f"max abs diff:  {diff.max().item():.3e}")
print(f"mean abs diff: {diff.mean().item():.3e}")
assert torch.allclose(hooked, ref, atol=1e-5, rtol=1e-4), "MISMATCH"
print("✓ smoke test passed")

max abs diff:  1.192e-07
mean abs diff: 3.386e-09
✓ smoke test passed
